In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator
from pandas.plotting import scatter_matrix

In [ ]:
# Helper Functions
def pca_func(data: pd.DataFrame, title_suffix: str = "") -> PCA:
    """
    Perform PCA and plot explained variance.
    Returns the fitted PCA model and the PCA scores DataFrame.
    """
    pca = PCA(n_components=data.shape[1])
    scores = pca.fit_transform(data)

    # Plot explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, min(10, data.shape[1])+1), pca.explained_variance_[:min(10, data.shape[1])], marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()


    # Print summary
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)

    df_scores = pd.DataFrame(
        scores,
        columns=[f'PC{i}' for i in range(1, scores.shape[1] + 1)],
        index=data.index
    )
    return pca, df_scores

In [ ]:
def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,  labels: pd.Series = None, title: str = "Biplot") -> None:
    """
    Create a PCA biplot with score points and variable loadings.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scatter plot for the scores
    ax.scatter(df_scores.PC1.values, df_scores.PC2.values, color='b', alpha=0.7)
    
    # Axis labeling
    expl_var = 100 * pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({expl_var[0]:.1f}% explained var.)", fontsize=10)
    ax.set_ylabel(f"PC2 ({expl_var[1]:.1f}% explained var.)", fontsize=10)
    
    # Determine which labels to use
    if labels is None:
        labels_to_use = df_scores.index
    else:
        labels_to_use = labels
    
    # Label sample points
    for i, name in enumerate(labels_to_use):
        x = df_scores.iloc[i, 0]
        y = df_scores.iloc[i, 1]
        ax.text(x, y, str(name), fontsize=9, color='blue', alpha=0.8)
        
    # Secondary axes for loadings
    ax2 = ax.twinx().twiny()
    
    font = {'color': 'g', 'weight': 'bold', 'size': 10}
    
    # Plot loading vectors and labels
    for col in df_loadings.columns.values:
        tipx = df_loadings.loc['PC1', col]
        tipy = df_loadings.loc['PC2', col]
        ax2.arrow(0, 0, tipx, tipy, color='r', alpha=0.5)
        ax2.text(tipx * 1.07, tipy * 1.07, col, fontdict=font, ha='center', va='center')
    
    # Align axes centers
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    
    # Keep square aspect ratio for accurate geometry
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')
    
    plt.title(title)
    plt.tight_layout()

In [ ]:
# -----------------------------
# Load and Prepare Data
# -----------------------------
dfr = pd.read_csv("data/Voetballers.csv", index_col=0, encoding="latin1")
print("Original Data Preview:\n", dfr.head())


In [ ]:
# Keep only numeric columns (remove Player, Team, Position)
df = dfr.select_dtypes(include=[np.number])
print("\nNumeric Data Preview:\n", df.head())

# Standardize data
dfs = pd.DataFrame(
    StandardScaler().fit_transform(df),
    columns=df.columns,
    index=df.index
)

In [ ]:
# Inspect the data:
scatter_matrix(df.iloc[:,:9], alpha=0.6, figsize=(8, 8), diagonal='hist')
plt.suptitle("Scatterplot Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# PCA and Biplot
# -----------------------------
pca, df_scores = pca_func(dfs, " (Voetballers)")
df_loadings = pd.DataFrame(pca.components_, columns=df.columns, index=df_scores.columns)


In [ ]:
biplot(df_scores, df_loadings, pca, title="Voetballers - PCA Biplot")

In [ ]:
# -----------------------------
# PCA with Varimax Rotation
# -----------------------------
k=3
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(dfs)

loadings_df = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2", "PC3"])
print(loadings_df)

rotator = Rotator(method="varimax")
loadings_rot = rotator.fit_transform(pca.components_.T)

R = rotator.rotation_

load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["RPC1", "RPC2", "RPC3"])
print(load_rot_df)

In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than this value (cut),  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)

print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))
